In [1]:
# Imports
import boto3
from sagemaker import get_execution_role
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
role = get_execution_role()
bucket = 'json-avalos-ads508'
delog_data_key = 'Delivery_Logistics 2.csv'
dysulog_data_key = 'dynamic_supply_chain_logistics_dataset.csv'
trade_cus_data_key = 'trade_customs_dataset 2.csv'
logistics_data_key = 'Logistics_supply_chain.csv'
del_loc = 's3://{}/{}'.format(bucket, delog_data_key)
dyn_loc = 's3://{}/{}'.format(bucket, dysulog_data_key)
trade_loc = 's3://{}/{}'.format(bucket, trade_cus_data_key)
logistics_loc = 's3://{}/{}'.format(bucket, logistics_data_key)

In [3]:
!pip install openpyxl # dependency
delivery = pd.read_csv(del_loc)
dynamic = pd.read_csv(dyn_loc)
trade = pd.read_csv(trade_loc)
logistics = pd.read_csv(logistics_loc)

In [4]:
def clean_trade(df: pd.DataFrame) -> pd.DataFrame:
    """trade_customs_dataset — parse dates, derive delay columns, basic QA."""
    df = df.copy()
    # Parse datetime columns
    for col in ["Shipment_Date", "Estimated_Arrival_Date", "Actual_Arrival_Date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    # Drop rows where we can't compute the delay
    before = len(df)
    df = df.dropna(subset=["Estimated_Arrival_Date", "Actual_Arrival_Date"])
    print(f"  trade: dropped {before - len(df)} rows with missing arrival dates")
    # Derived delay features
    df["Transit_Delay_Days"] = (
        df["Actual_Arrival_Date"] - df["Estimated_Arrival_Date"]
    ).dt.days
    df["Is_Delayed"] = (df["Transit_Delay_Days"] > 0).astype(int)
    # Total delay = transit + customs
    if "Customs_Delay_Days" in df.columns:
        df["Total_Delay_Days"] = df["Transit_Delay_Days"] + df["Customs_Delay_Days"].fillna(0)
    # Temporal features
    df["Shipment_Month"] = df["Shipment_Date"].dt.month
    df["Shipment_DayOfWeek"] = df["Shipment_Date"].dt.dayofweek   # 0=Mon
    df["Shipment_IsWeekend"] = (df["Shipment_DayOfWeek"] >= 5).astype(int)
    # Normalise carrier / route strings
    for col in ["Carrier_Name", "Route_Code"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()
    return df


def clean_delivery(df: pd.DataFrame) -> pd.DataFrame:
    """Delivery_Logistics — sanity checks, derived efficiency features."""
    df = df.copy()
 
    # Normalise all column names to lowercase for consistent access
    df.columns = df.columns.str.strip().str.lower()
 
    print(f"  delivery: columns found → {list(df.columns)}")
    print(f"  delivery: {len(df):,} rows on load")
 
    # ── Parse time columns — handle multiple possible formats ──────────────────
    def parse_time_to_hours(series: pd.Series, col: str) -> pd.Series:
        """
        Convert a time column to float hours regardless of format:
          "02:30"      -> 2.5
          "2:30:00"    -> 2.5
          "2.5"        -> 2.5
          "2.5 hrs"    -> 2.5
          "150"        -> 150.0  (assumed already in minutes if >24, else hours)
        """
        # Try plain numeric first
        attempt = pd.to_numeric(series, errors="coerce")
        if attempt.notna().mean() > 0.5:
            print(f"  delivery: '{col}' parsed as plain numeric")
            return attempt
 
        # Try HH:MM or HH:MM:SS string format
        def hhmm_to_hours(val):
            try:
                parts = str(val).strip().split(":")
                if len(parts) >= 2:
                    return int(parts[0]) + int(parts[1]) / 60
            except Exception:
                pass
            # Strip non-numeric suffixes like "hrs", "h", "min"
            try:
                import re
                cleaned = re.sub(r"[^\d.]", "", str(val))
                return float(cleaned) if cleaned else float("nan")
            except Exception:
                return float("nan")
 
        result = series.apply(hhmm_to_hours)
        n_converted = result.notna().sum()
        print(f"  delivery: '{col}' parsed from string format — "
              f"{n_converted:,}/{len(series):,} values converted, "
              f"sample raw values: {series.dropna().head(3).tolist()}")
        return result
 
    time_cols = ["expected_time_hours", "delivery_time_hours"]
    for col in time_cols:
        if col in df.columns:
            df[col] = parse_time_to_hours(df[col], col)
 
    # Coerce remaining numeric columns
    other_numeric = ["distance_km", "delivery_cost", "package_weight_kg", "delivery_rating"]
    for col in other_numeric:
        if col in df.columns:
            before_nulls = df[col].isna().sum()
            df[col] = pd.to_numeric(df[col], errors="coerce")
            new_nulls = df[col].isna().sum() - before_nulls
            if new_nulls > 0:
                print(f"  delivery: coerced {new_nulls} non-numeric values to NaN in '{col}'")
 
    # Guard against zero / null denominators — only filter if columns exist
    if "distance_km" in df.columns:
        before = len(df)
        df = df[df["distance_km"].notna() & (df["distance_km"] > 0)].copy()
        print(f"  delivery: dropped {before - len(df)} rows where distance_km <= 0 or null")
    else:
        print("  ⚠ delivery: 'distance_km' column not found — skipping filter")
 
    if "expected_time_hours" in df.columns:
        before = len(df)
        df = df[df["expected_time_hours"].notna() & (df["expected_time_hours"] > 0)].copy()
        print(f"  delivery: dropped {before - len(df)} rows where expected_time_hours <= 0 or null")
    else:
        print("  ⚠ delivery: 'expected_time_hours' column not found — skipping filter")
 
    # Derived features (only if source columns exist)
    if "delivery_time_hours" in df.columns and "expected_time_hours" in df.columns:
        df["time_vs_expected"] = df["delivery_time_hours"] / df["expected_time_hours"]
 
    if "delivery_cost" in df.columns and "distance_km" in df.columns:
        df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]
 
    # Standardise the delay flag name (some versions use 'delayed', 'Delayed', 'is_delayed')
    delay_col = next((c for c in df.columns if c.lower() in ("delayed", "is_delayed")), None)
    if delay_col and delay_col != "delayed":
        df = df.rename(columns={delay_col: "delayed"})
 
    print(f"  delivery: {len(df):,} rows after cleaning, {df.shape[1]} cols")
    return df


def clean_dynamic(df: pd.DataFrame) -> pd.DataFrame:
    """dynamic_supply_chain_logistics_dataset — cast types, derive features."""
    df = df.copy()
    # Ensure congestion columns are numeric integers
    for col in ["port_congestion_level", "traffic_congestion_level"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    # Combined congestion index
    df["combined_congestion"] = (
        df.get("port_congestion_level", 0) + df.get("traffic_congestion_level", 0)
    )
    # Risk × clearance interaction
    if "route_risk_level" in df.columns and "customs_clearance_time" in df.columns:
        df["risk_clearance_product"] = (
            df["route_risk_level"] * df["customs_clearance_time"]
        )
    # Log-transform shipping costs (right-skewed)
    if "shipping_costs" in df.columns:
        df["log_shipping_costs"] = np.log1p(df["shipping_costs"])
    return df


def _iqr_bounds(series: pd.Series, k: float = 3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def clean_logistics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── 1. Timestamp parsing + time features ─────────────────────────────────
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    df["year"] = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["day_of_week"] = df["timestamp"].dt.dayofweek   # 0 = Monday
    df["hour"] = df["timestamp"].dt.hour

    # ── 2. Binary target ──────────────────────────────────────────────────────
    df["is_delayed"] = (df["delivery_time_deviation"] > 0).astype(int)

    # ── 3. Operational risk indicators (thresholds from team notebook) ────────
    df["high_port_congestion"] = (df["port_congestion_level"] >= 7).astype(int)
    df["bad_weather"] = (df["weather_condition_severity"] >= 0.7).astype(int)
    df["low_supplier_reliability"]= (df["supplier_reliability_score"] <= 0.3).astype(int)

    # ── 4. Select final feature set ───────────────────────────────────────────
    keep = [
        "timestamp",
        "lead_time_days",
        "weather_condition_severity",
        "port_congestion_level",
        "supplier_reliability_score",
        "delivery_time_deviation",
        "year",
        "month",
        "day",
        "day_of_week",
        "hour",
        "is_delayed",
        "high_port_congestion",
        "bad_weather",
        "low_supplier_reliability",
    ]
    missing = [c for c in keep if c not in df.columns]
    if missing:
        print(f"  ⚠ logistics: expected columns not found: {missing}")
    df = df[[c for c in keep if c in df.columns]].copy()

    # ── 5. Validate ───────────────────────────────────────────────────────────
    print(f"  logistics: {len(df):,} rows, {df.shape[1]} cols, "
          f"is_delayed rate = {df['is_delayed'].mean():.2%}")
    n_missing = df.isnull().sum().sum()
    if n_missing:
        print(f"  logistics: {n_missing} nulls remaining (check timestamp parse)")

    return df

In [5]:
trade = clean_trade(trade)
delivery = clean_delivery(delivery)
dynamic = clean_dynamic(dynamic)
logistics = clean_logistics(logistics)

  trade: dropped 0 rows with missing arrival dates
  delivery: columns found → ['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost']
  delivery: 25,000 rows on load
  delivery: 'expected_time_hours' parsed from string format — 25,000/25,000 values converted, sample raw values: ['1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000003', '1970-01-01 00:00:00.000000016']
  delivery: 'delivery_time_hours' parsed from string format — 25,000/25,000 values converted, sample raw values: ['1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000002', '1970-01-01 00:00:00.000000010']
  delivery: dropped 0 rows where distance_km <= 0 or null
  delivery: dropped 0 rows where expected_time_hours <= 0 or null
  delivery: 25,000 rows after cleaning, 17 cols
  logistics: 32,065 rows, 1

In [6]:
delivery.shape

(25000, 17)